In [3]:
import requests
import re
from rdflib import Graph, URIRef, RDF, Namespace
from typing import List, Dict, Set, Any
import pyshacl # ⬅️ CORRECTED IMPORT

# --- CONFIGURATION ---
SHACL_RULES_URL = "https://raw.githubusercontent.com/DOI-DO/dcat-us/main/shacl/dcat-us_3.0_shacl_shapes.ttl"
AGENCY_DATA_URL = "https://ngda-transportation-geoplatform.hub.arcgis.com/api/feed/dcat-us/1.1?id=23d91bd988ac4fc9943128965bddfa37"

# --- Define Namespaces ---
SH = Namespace("http://www.w3.org/ns/shacl#")
DCAT = Namespace("http://www.w3.org/ns/dcat#")
DCTERMS = Namespace("http://purl.org/dc/terms/")
RDFS = Namespace("http://www.w3.org/2000/01/rdf-schema#")

# Define the required URIs for entity lookup (Classes)
DCAT_CATALOG = DCAT.Catalog
DCAT_DATASET = DCAT.Dataset


def fetch_and_load_graph(url: str, format: str) -> Graph:
    """Fetches content from a URL and loads it into an RDF graph."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        graph = Graph()
        graph.parse(data=response.text, format=format)
        return graph
    except Exception:
        return None


def get_properties_from_shacl(shacl_graph: Graph, target_class: URIRef) -> List[Dict]:
    """Dynamically queries SHACL graph for ALL properties (M and R) with definitions and status."""
    properties = []
    
    shapes = list(shacl_graph.subjects(SH.targetClass, target_class))
    
    for shape in shapes:
        for prop_node in shacl_graph.objects(shape, SH.property):
            
            prop_uri_node = shacl_graph.value(prop_node, SH.path)
            if not prop_uri_node: continue
            
            min_count = shacl_graph.value(prop_node, SH.minCount)
            status = "Mandatory (M)" if min_count and int(min_count) > 0 else "Recommended (R)"
            
            label = shacl_graph.value(prop_uri_node, RDFS.label)
            comment = shacl_graph.value(prop_node, RDFS.comment) or shacl_graph.value(prop_uri_node, RDFS.comment)
            
            properties.append({
                'PropertyURI': str(prop_uri_node),
                'SimpleName': str(label) if label else get_clean_uri(prop_uri_node),
                'Definition': str(comment) if comment else "Definition not found in SHACL file.",
                'Status': status
            })
                    
    return properties


def get_clean_uri(uri_node: Any) -> str:
    """Dynamically simplifies any URI by returning the last segment."""
    uri_str = str(uri_node)
    if uri_str.startswith('N'): return "BNode_" + uri_str[1:8]
    if '#' in uri_str: return uri_str.split('#')[-1]
    return uri_str.split('/')[-1]


def clean_property_uri_for_summary(finding: Dict, defs: List[Dict]) -> str:
    """Heuristically maps an ambiguous violation message to the correct simple property name."""
    
    match_uri = finding['PropertyURI']
    match = next((d for d in defs if d['PropertyURI'] == match_uri), None)
    if match:
        return match['SimpleName']

    message_text = finding['Message'].lower()
    
    for d in defs:
        simple_name = d['SimpleName'].lower().replace(' ', '')
        if simple_name in message_text:
            return d['SimpleName']
            
    return finding['PropertyURI'].split('/')[-1].split('#')[-1]


def analyze_and_report(data_url: str, shacl_url: str):
    
    # 1. Load Graphs and Run Validation
    print("1. Loading DCAT-US 3.0 SHACL Rules...")
    shacl_graph = fetch_and_load_graph(shacl_url, 'turtle')
    
    print(f"2. Loading Target Data Graph from {data_url}...")
    data_graph = fetch_and_load_graph(data_url, 'json-ld')
    
    if not shacl_graph or not data_graph: return

    # 2. DYNAMIC ANALYSIS: Get Conceptual Requirements
    catalog_mandatory_defs = get_properties_from_shacl(shacl_graph, DCAT_CATALOG)
    dataset_mandatory_defs = get_properties_from_shacl(shacl_graph, DCAT_DATASET)
    
    print("3. Running Comprehensive SHACL validation...")
    
    # ⬅️ CORRECTED FUNCTION CALL: Use pyshacl.validate
    conforms, results_graph, results_text = pyshacl.validate(
        data_graph, shacl_graph=shacl_graph, inference='rdfs', abort_on_error=False
    )
    
    if conforms:
        print("\n🎉 The catalog meets all DCAT-US 3.0 Mandatory requirements.")
        return

    # 3. Process Validation Results
    query = """
    SELECT ?focusNode ?severity ?message ?sourceShape
    WHERE {
        ?result a sh:ValidationResult .
        ?result sh:focusNode ?focusNode .
        ?result sh:resultSeverity ?severity .
        ?result sh:resultMessage ?message .
        ?result sh:sourceShape ?sourceShape .
    }
    """
    
    severity_map = {
        str(URIRef("http://www.w3.org/ns/shacl#Violation")): "MANDATORY",
        str(URIRef("http://www.w3.org/ns/shacl#Warning")): "RECOMMENDED"
    }
    
    findings: List[Dict] = []
    
    for row in results_graph.query(query):
        focus_node = row.focusNode
        
        # Determine the TYPE of the entity that failed
        entity_type = "Supporting Entity"
        if (focus_node, RDF.type, DCAT_CATALOG) in data_graph:
            entity_type = "Catalog"
        elif (focus_node, RDF.type, DCAT_DATASET) in data_graph:
            entity_type = "Dataset"
        
        property_uri = str(row.message).split('on property')[-1].strip() if 'on property' in str(row.message) else 'Structural Constraint'
        message = str(row.message).split('on property')[0].strip()
        
        findings.append({
            'Severity': severity_map.get(str(row.severity), "INFO"),
            'EntityType': entity_type,
            'PropertyURI': property_uri,
            'Message': message,
        })
        
    output_markdown_guide(findings, catalog_mandatory_defs, dataset_mandatory_defs)


def output_markdown_guide(findings: List[Dict], catalog_defs: List[Dict], dataset_defs: List[Dict]):
    """Generates the sequential, user-friendly markdown conversion guide with definitions."""
    
    mandatory_findings = [f for f in findings if f['Severity'] == 'MANDATORY']
    
    # --- TOP-LEVEL SUMMARY ---
    print("\n" + "="*80)
    print("## 🚨 Top-Level Conversion Summary (MANDATORY FIXES)\n")
    print("This summary lists all critical gaps found. Fix these first to achieve v3.0 compliance.")
    print("| Status | Entity Type | Missing/Invalid Property | Required Action Summary |")
    print("| :--- | :--- | :--- | :--- |")
    
    all_defs = catalog_defs + dataset_defs
    
    for f in mandatory_findings:
        
        simple_name = clean_property_uri_for_summary(f, all_defs)

        action_summary = "Structural Failure"
        if 'dcat:dataset' in f['PropertyURI']:
            action_summary = "MISSING CORE ENTITY: The Catalog is empty (no datasets found)."
        elif 'Less than 1 values on' in f['Message']:
            action_summary = f"Missing mandatory property: {simple_name}"
        elif 'does not conform' in f['Message']:
            action_summary = f"STRUCTURAL ERROR: Nested entity ({simple_name}) is malformed."
             
        print(f"| 🛑 | **{f['EntityType']}** | `{simple_name}` | **{action_summary}** |")

    print("\n" + "="*80)
    
    # --- PART 1: FIX THE CATALOG (TOP-LEVEL) ---
    print("\n## 🛠️ Part 1: Fix the Catalog (Top-Level Compliance)\n")
    print("This section details the gaps on the `dcat:Catalog` entity, which represents the `data.json` file itself.")
    
    print("### 🛑 Mandatory Catalog Gaps\n")
    print("| Requirement | Property URI | Definition | Status | Actionable Task |")
    print("| :--- | :--- | :--- | :--- | :--- |")
    
    catalog_mandatory_findings = [f for f in mandatory_findings if f['EntityType'] == 'Catalog']
    
    for definition in catalog_defs:
        uri = definition['PropertyURI']
        violation = next((f for f in catalog_mandatory_findings if uri == f['PropertyURI']), None)
        
        status = "OK ✅"
        task = "No action required."

        if violation:
            status = "MISSING 🛑"
            
            if 'dcat:dataset' in uri:
                task = "The Catalog is EMPTY. You MUST add at least one dcat:Dataset entity."
            elif 'Less than 1 values on' in violation['Message']:
                task = f"Add the missing mandatory property: '{definition['SimpleName']}'."
            elif 'does not conform' in violation['Message']:
                task = f"The value for this property (e.g., Publisher Contact) has an invalid structure. Fix the sub-fields of the nested entity."

        print(f"| **Mandatory** | `{definition['SimpleName']}` | {definition['Definition']} | **{status}** | {task} |")

    # --- PART 2: FIX THE DATASETS (CONTENT) ---
    print("\n---\n")
    print("## 🛠️ Part 2: Fix the Datasets (Content Compliance)\n")
    
    if any('dcat:dataset' in f['PropertyURI'] for f in catalog_mandatory_findings):
        print("🛑 **ATTENTION:** Your catalog failed Part 1 because it is empty. The following list details the MINIMUM REQUIREMENTS that must be included for EVERY dataset you create.\n")
    
    print("### 🛑 Mandatory Dataset Requirements\n")
    print("| Simple Name | Property URI | Definition |")
    print("| :--- | :--- | :--- |")

    for definition in dataset_defs:
        print(f"| **{definition['SimpleName']}** | `{definition['PropertyURI'].split('/')[-1].split('#')[-1]}` | {definition['Definition']} |")

    print("\n" + "="*80)


# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    analyze_and_report(AGENCY_DATA_URL, SHACL_RULES_URL)

1. Loading DCAT-US 3.0 SHACL Rules...
2. Loading Target Data Graph from https://ngda-transportation-geoplatform.hub.arcgis.com/api/feed/dcat-us/1.1?id=23d91bd988ac4fc9943128965bddfa37...


Usage of abort_on_error is deprecated. Use abort_on_first instead.


3. Running Comprehensive SHACL validation...

## 🚨 Top-Level Conversion Summary (MANDATORY FIXES)

This summary lists all critical gaps found. Fix these first to achieve v3.0 compliance.
| Status | Entity Type | Missing/Invalid Property | Required Action Summary |
| :--- | :--- | :--- | :--- |
| 🛑 | **Catalog** | `catalog` | **Missing mandatory property: catalog** |
| 🛑 | **Catalog** | `catalog` | **Missing mandatory property: catalog** |
| 🛑 | **Catalog** | `catalog` | **Missing mandatory property: catalog** |
| 🛑 | **Catalog** | `catalog` | **Missing mandatory property: catalog** |


## 🛠️ Part 1: Fix the Catalog (Top-Level Compliance)

This section details the gaps on the `dcat:Catalog` entity, which represents the `data.json` file itself.
### 🛑 Mandatory Catalog Gaps

| Requirement | Property URI | Definition | Status | Actionable Task |
| :--- | :--- | :--- | :--- | :--- |
| **Mandatory** | `accessRights` | Definition not found in SHACL file. | **OK ✅** | No action required. |
| *